# Idiap / USSD teachers: 188 participant internal 60:20:20 protocol, seed 103

Run cells in order. This notebook starts with the **split and provenance audit**. It combines the 107 original TRAIN participants, 34 usable DEV participants (440 excluded), and 47 locally labeled original TEST participants into a new, stratified, participant-disjoint TRAIN-113 / VAL-37 / internal TEST-38 split. Internal TEST labels are used solely to stratify and freeze membership. Their features and predictions must remain unopened until the final evaluation.

**Checkpoint rule:** The published Idiap and USSD checkpoints already saw some participants who may now be in VAL or internal TEST. Do not reuse those weights for validation or final scoring. The requested transfer-learning experiments load the original checkpoints and fine-tune on TRAIN-113; their VAL and TEST results are exploratory because of prior participant exposure. This is an internal redistributed split, not an official AVEC test result. The provenance of local `full_test_split.csv` labels has not been independently verified. Earlier results on other splits cannot be compared as if this were an untouched test.

Stage 1 below saves only the split. We will add preprocessing/teacher stages to this same notebook after the split audit is confirmed. The older experiment folders are left untouched.

**Protocol revision:** This 60:20:20 split follows inspection of earlier 150/19/19 validation results. It retains all 19 previously closed test participants in the 38-person test, moves the previous 19-person validation group into training, and draws 19 more test plus 37 validation participants from the previous training group. The additional test participants appeared in earlier training runs. Treat this as a revised exploratory protocol; do not claim its test is pristine across all research iterations. Previous experiment outputs remain intact.


## 1. Mount Drive and load the pinned GitHub branch


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
if not (Path('/content/drive/MyDrive/DAIC_WOZ') / 'metadata').is_dir():
    drive.mount('/content/drive')
import base64, os, subprocess, sys
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert (DAIC / 'metadata').is_dir(), f'Drive unavailable; reconnect Drive in Colab, then rerun: {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/official141-original-teachers-80-10-10'
ROOT = Path('/content/reliability-aware-depression-kd')
token = userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN to Colab Secrets'
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy(); env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {header}')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True, env=env)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=env)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
CODE = Path('/content/reliability-aware-depression-kd-original188')
if not CODE.exists():
    subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
else:
    assert (CODE / '.git').exists(), f'Existing folder is not a Git worktree: {CODE}'
    # Git refuses the switch if it would overwrite tracked or conflicting untracked work.
    subprocess.run(['git', '-C', str(CODE), 'switch', '--detach', revision], check=True)
assert subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() == revision
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')], check=True)
del token, header, env
# Colab retains already imported modules from earlier revisions.
sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
sys.path.insert(0, str(CODE))
for name in list(sys.modules):
    if name == 'scripts' or name.startswith('scripts.'):
        del sys.modules[name]
assert (CODE / 'scripts/experiments/internal188/coverage.py').is_file(), 'Checkout is missing coverage.py'
print('Revision:', revision)

def refresh_code():
    """Update this existing worktree; no new /content code folder."""
    secret = userdata.get('GITHUB_TOKEN')
    assert secret, 'GITHUB_TOKEN missing from Colab Secrets'
    credentials = os.environ.copy()
    credentials.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
                       GIT_CONFIG_VALUE_0='AUTHORIZATION: basic ' +
                       base64.b64encode(f'x-access-token:{secret}'.encode()).decode())
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=credentials)
    latest = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    current = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
    if latest != current:
        # Git itself refuses a switch that would overwrite local work.
        subprocess.run(['git', '-C', str(CODE), 'switch', '--detach', latest], check=True)
    sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
    sys.path.insert(0, str(CODE))
    for name in list(sys.modules):
        if name == 'scripts' or name.startswith('scripts.'):
            del sys.modules[name]
    print('Code revision:', latest)
    del secret, credentials


## 2. Freeze the 113 / 37 / 38 split


In [ ]:
import pandas as pd
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from scripts.experiments.internal188.split import main as split_stage
PRIOR_EXP = DAIC / 'experiments' / 'idiap_ussd_internal188_seed103'
EXP = DAIC / 'experiments' / 'idiap_ussd_internal188_60_20_20_seed103'
split_stage(['--daic-root', str(DAIC), '--output', str(EXP / 'split'), '--prior-split-dir', str(PRIOR_EXP / 'split'), '--seed', '103'])
manifest = pd.read_csv(EXP / 'split' / 'manifest.csv')
assert len(manifest) == 188 and manifest.participant_id.nunique() == 188
assert 440 not in set(manifest.participant_id)
assert manifest.split.value_counts().to_dict() == {'train': 113, 'val': 37, 'student_test': 38}
prior = pd.read_csv(PRIOR_EXP / 'split' / 'manifest.csv')
assert set(prior.loc[prior.split.eq('student_test'), 'participant_id']) <= set(manifest.loc[manifest.split.eq('student_test'), 'participant_id'])
assert set(prior.loc[prior.split.eq('val'), 'participant_id']) <= set(manifest.loc[manifest.split.eq('train'), 'participant_id'])
print(pd.crosstab(manifest.split, manifest.label))
print('Source audit:', manifest.groupby(['split', 'source_split']).size().to_dict())
print('Saved:', EXP / 'split' / 'manifest.csv')


## 3. Audit TRAIN/VAL transcripts and ComParE16 cache

This reads audio feature arrays and transcript paths for **150 TRAIN/VAL participants only**. It does not open internal TEST recordings, transcripts or feature arrays. Existing feature caches may be shared; keep their directories available. If coverage is incomplete, stop and report the exact missing participant IDs before any training.


In [ ]:
from scripts.experiments.internal188.coverage import main as coverage_stage
cache_candidates = [DAIC / 'experiments' / 'ussd_compare16',
                    DAIC / 'experiments' / 'full188_seed42' / 'features',
                    DAIC / 'experiments' / 'full188_80_10_10_seed103' / 'features']
available = [p for p in cache_candidates if (p / 'participant_manifest.csv').is_file()]
print('Available feature caches:', [str(p) for p in available])
assert available, 'No ComParE16 feature manifest found'
coverage_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                '--output', str(EXP / 'coverage'),
                *[arg for p in available for arg in ('--feature-cache', str(p))]])
covered = pd.read_csv(EXP / 'coverage' / 'participant_manifest.csv')
assert covered.split.value_counts().to_dict() == {'train': 113, 'val': 37}
print('Coverage:', pd.crosstab(covered.split, covered.label))


## 4. Idiap checkpoint initialization and new fine-tuning

Run this after sections 1–3. It downloads the **exact published checkpoint and vectorizer**, verifies their Git blob hashes, builds a new TRAIN-113 graph and vocabulary, then transfers the output layer and first-layer columns for shared vocabulary tokens. Newly introduced tokens keep random initialization. It then fine-tunes a new teacher on TRAIN-113 and selects the checkpoint with VAL-37 macro F1. The published checkpoint already trained on some current VAL and internal TEST participants; this run is exploratory and cannot support a clean held-out test claim. Its output directory is separate from the fresh-weight experiment.


In [ ]:
refresh_code()
assert (CODE / 'scripts/experiments/internal188/idiap_source.py').is_file(), 'Stale CODE checkout: rerun notebook section 1 before training'
from scripts.experiments.internal188.idiap_source import ensure_source, ensure_published_weights
from scripts.experiments.internal188.train_text_teacher import main as text_teacher_stage
idiap_source = ensure_source(DAIC.parent / 'tools' / 'solo_teacher_sources' / 'bias_in_daic-woz')
checkpoint, old_vectorizer = ensure_published_weights()
text_teacher_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                    '--coverage-dir', str(EXP / 'coverage'),
                    '--idiap-source', str(idiap_source),
                    '--published-checkpoint', str(checkpoint),
                    '--published-vectorizer', str(old_vectorizer),
                    '--output', str(EXP / 'teachers' / 'text_published_init'), '--seed', '103'])


In [ ]:
import json
pretrained_audit = json.loads((EXP / 'teachers' / 'text_published_init' / 'audit.json').read_text())
print('Transferred:', pretrained_audit['initialization'])
print('Prior exposure:', pretrained_audit['published_checkpoint_prior_exposure'])
print('Best epoch:', pretrained_audit['best_epoch'])
print('TRAIN:', {k: pretrained_audit['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: pretrained_audit['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})


## 5. Train USSD audio teacher with the published run #4 weights

This starts from the exact pinned `md_35_epochs.pth`, uses the **pinned normalization saved with that checkpoint**, and trains a new model using 8 sampled 384-frame segments per participant with a participant-level weighted loss. It prints TRAIN and VAL macro F1 each epoch. Epoch 0 is evaluated and remains eligible if fine-tuning degrades VAL macro F1. This adaptation objective is different from the author's original speaker-disentanglement objective. The original run #4 weights had prior exposure to 11 current VAL and 14 current internal TEST participants, so this result is exploratory. Internal TEST is never inferred.


In [ ]:
refresh_code()
from scripts.teachers.ussd_audio.common import _resolve_pinned_author_artifact, AUTHOR_RUN4_CHECKPOINT_SHA256
from scripts.teachers.ussd_audio import CHECKPOINT_NAME
from scripts.experiments.internal188.train_audio_teacher import main as audio_teacher_stage
ussd_checkpoint = _resolve_pinned_author_artifact(
    DAIC.parent / 'tools' / 'solo_teacher_sources' / 'USSD-depression',
    CHECKPOINT_NAME, AUTHOR_RUN4_CHECKPOINT_SHA256)
audio_teacher_stage(['--split-dir', str(EXP / 'split'), '--coverage-dir', str(EXP / 'coverage'),
                     '--features', str(EXP / 'coverage'), '--published-checkpoint', str(ussd_checkpoint),
                     '--author-root', str(DAIC.parent / 'tools' / 'solo_teacher_sources' / 'USSD-depression'),
                     '--output', str(EXP / 'teachers' / 'audio_published_author_norm_v2'), '--seed', '103'])


In [ ]:
import json
audio_audit = json.loads((EXP / 'teachers' / 'audio_published_author_norm_v2' / 'audit.json').read_text())
print('Epoch 0 VAL:', {k: audio_audit['epoch_zero']['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('Audio best epoch:', audio_audit['best_epoch'])
print('Prior exposure:', audio_audit['published_checkpoint_prior_exposure'])
print('TRAIN:', {k: audio_audit['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: audio_audit['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})


## 6. Audit participant aggregation on the saved audio checkpoint

No further training. Score TRAIN and VAL segments from the selected published-initialized checkpoint using both mean segment probability and the author's hard segment majority vote. Confirm the recalculated mean probabilities match the saved teacher targets. The internal TEST is never opened.


In [ ]:
refresh_code()
from scripts.experiments.internal188.audit_audio_aggregation import main as audit_audio_aggregation
AUDIT = EXP / 'teachers' / 'audio_aggregation_audit_v2'
audit_audio_aggregation(['--split-dir', str(EXP / 'split'),
                         '--coverage-dir', str(EXP / 'coverage'),
                         '--teacher-dir', str(EXP / 'teachers' / 'audio_published_author_norm_v2'),
                         '--output', str(AUDIT)])


## 7. Clean audio experiment: fresh USSD architecture and balanced participant pairs

This is the independent audio teacher candidate for the 188-person split. The ComParE16 arrays remain label-independent, while normalization is fitted on TRAIN-113. No published DAIC-trained checkpoint or original author normalization is loaded. Every TRAIN batch samples one depressed and one non-depressed participant, with 8 segments each; loss is computed once per participant, without an extra class weight. Checkpoints are selected on VAL-37 macro F1. Internal TEST is closed. This is a new training objective using the USSD architecture, not a reproduction of the full author speaker-disentanglement loss.


In [ ]:
refresh_code()
from scripts.experiments.internal188.train_audio_teacher import main as audio_teacher_stage
CLEAN_AUDIO = EXP / 'teachers' / 'audio_fresh_balanced_v1'
audio_teacher_stage(['--split-dir', str(EXP / 'split'), '--coverage-dir', str(EXP / 'coverage'),
                     '--features', str(EXP / 'coverage'), '--output', str(CLEAN_AUDIO),
                     '--seed', '103', '--balanced-pairs', '--lr', '0.001'])


In [ ]:
import json
clean_audio = json.loads((CLEAN_AUDIO / 'audit.json').read_text())
print('Best epoch:', clean_audio['best_epoch'])
print('TRAIN:', {k: clean_audio['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: clean_audio['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('Provenance:', clean_audio['normalization_source'], clean_audio['published_checkpoint_prior_exposure'])


## 8A. Cache externally pretrained WavLM Base+ audio embeddings

This is the next audio teacher candidate. WavLM Base+ is pretrained outside DAIC-WOZ; **its weights stay frozen**. For each TRAIN or VAL participant, recreate participant-only speech using the original USSD transcript interval procedure, resample to 16 kHz, select eight evenly spaced 6-second crops, and store the mean and standard deviation of WavLM embeddings. The exact model revision and input hashes are recorded. Completed participant caches are reused after interruption. No internal TEST audio is opened. The first pass downloads the model and can take considerable time.


In [ ]:
refresh_code()
import os
os.environ['HF_HOME'] = str(EXP / 'models' / 'huggingface_cache')
from scripts.experiments.internal188.wavlm_features import main as wavlm_features_stage
WAVLM = EXP / 'features' / 'wavlm_base_plus_v1'
wavlm_features_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                      '--coverage-dir', str(EXP / 'coverage'), '--output', str(WAVLM),
                      '--batch-size', '2',
                      '--reuse-from', str(PRIOR_EXP / 'features' / 'wavlm_base_plus_v1')])


## 8B. Fit a participant-level teacher on TRAIN-113 only

A predeclared balanced logistic classifier uses the cached 1,536-dimensional participant vectors. Its scaler and classifier are fitted only on TRAIN-113. VAL-37 is reported at the fixed 0.5 threshold, without threshold or hyperparameter search. It exports TRAIN in-sample KD logits/probabilities. Keep internal TEST closed.


In [ ]:
refresh_code()
from scripts.experiments.internal188.train_wavlm_teacher import main as wavlm_teacher_stage
WAVLM_TEACHER = EXP / 'teachers' / 'wavlm_base_plus_v1'
wavlm_teacher_stage(['--split-dir', str(EXP / 'split'),
                     '--coverage-dir', str(EXP / 'coverage'),
                     '--embeddings', str(WAVLM), '--output', str(WAVLM_TEACHER),
                     '--seed', '103'])


## Stop after WavLM validation

Share TRAIN and VAL metrics and confusion matrices. The internal test remains closed. We will only consider WavLM KD targets if validation shows independent signal.


## 9. Audit the public Step-Audio2 DAIC-WOZ LoRA adapter (exploratory)

This candidate is already trained on DAIC-WOZ. Download its small metadata first and record the exact revision and base-model declaration. The adapter publisher has not supplied the participant-level training roster, so we cannot establish that it did not see the current VAL or internal TEST participants. The original canonical TRAIN roster alone overlaps 23 current VAL and 23 current internal TEST participants. **Do not include this checkpoint in the main clean teacher/KD comparison or open internal TEST.** This stage does not run inference or alter the existing teachers.

Set `--download-weights` only after checking the metadata and resources. Its Step-Audio2 base model must be acquired separately; a LoRA adapter alone cannot make predictions.


In [ ]:
refresh_code()
from scripts.experiments.internal188.stepaudio2_adapter_audit import main as step_audio_audit
STEP_AUDIO = EXP / 'teachers' / 'stepaudio2_external_adapter_audit'
step_audio_audit(['--split-dir', str(EXP / 'split'), '--output', str(STEP_AUDIO)])
print((STEP_AUDIO / 'audit.json').read_text())


## 10. Run a one-person Step-Audio2 validation pilot

This cell installs `ms-swift` if needed, prepares one VAL participant, runs inference, streams the full Swift output, and saves `swift_infer.log` in the pilot folder. If Swift exits with an error, the last 35 log lines appear in the exception. It may download a large base model and require substantial GPU memory. Internal TEST is never opened.


In [ ]:
refresh_code()
import json, shutil, subprocess, sys
from scripts.experiments.internal188.prepare_stepaudio2_val import main as prepare_stepaudio2_val
from scripts.experiments.internal188.run_logged import run as run_logged
assert subprocess.check_output(['nvidia-smi', '-L'], text=True).strip(), 'A GPU runtime is required'
if shutil.which('swift') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ms-swift'])
assert shutil.which('swift'), 'ms-swift installation did not provide swift CLI'
STEP_PILOT = EXP / 'teachers' / 'stepaudio2_exploratory_val_pilot'
pilot_command = prepare_stepaudio2_val(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
    '--coverage-dir', str(EXP / 'coverage'), '--adapter-audit', str(STEP_AUDIO / 'audit.json'),
    '--output', str(STEP_PILOT), '--pilot-only'])
pilot_result = STEP_PILOT / 'pilot_predictions.jsonl'
if not pilot_result.is_file(): run_logged(pilot_command, STEP_PILOT / 'swift_infer.log', cwd=CODE)
assert pilot_result.is_file(), f'Inference produced no result: {pilot_result}'
pilot_rows = [json.loads(line) for line in pilot_result.read_text().splitlines() if line.strip()]
assert len(pilot_rows) == 1, f'Expected one pilot prediction, got {len(pilot_rows)}'
print('Raw pilot response:', pilot_rows[0])


## 11. Continue fine-tuning the released adapter (exploratory)

Run this only if section 10 produced a usable Yes/No prediction. It prepares 113 labeled TRAIN and 37 labeled VAL participant-only audio examples, then launches `swift sft` initialized from `maimai11/woz`. Checkpoints are saved under this experiment; internal TEST is closed. Validation loss selects checkpoints, but it is not participant macro F1. Prior DAIC exposure remains, so this run cannot support a clean holdout claim.


In [ ]:
refresh_code()
from scripts.experiments.internal188.prepare_stepaudio2_sft import main as prepare_stepaudio2_sft
assert (STEP_PILOT / 'pilot_predictions.jsonl').is_file(), 'Run section 10 first'
STEP_SFT = EXP / 'teachers' / 'stepaudio2_published_adapter_sft_exploratory'
sft_command = prepare_stepaudio2_sft(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
    '--coverage-dir', str(EXP / 'coverage'), '--adapter-audit', str(STEP_AUDIO / 'audit.json'),
    '--output', str(STEP_SFT)])
run_logged(sft_command, STEP_SFT / 'swift_sft.log', cwd=CODE)
